<a href="https://colab.research.google.com/github/nurfnick/NetworkScience/blob/main/HomeworkAssignments/Project3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# CS 5483 Network Science — Project 3
### Algorithms, Measurement Error, Real‑World Structure, and Random Graph Models (Instructions‑Only)

**Allowed libraries:** `networkx`, `numpy`, `matplotlib`. Additional packages require citation and justification in the write‑up.
**Students must write all code.**



## Dataset(s)
- Choose one **real network** (5k–50k edges recommended). Cite the source.
- Generate synthetic graphs in later parts for comparison.

**Show here:**
`|V|`, `|E|`, density, component summary, and any preprocessing choices.


In [ ]:

# TODO: Imports (NetworkX, NumPy, Matplotlib)
# import networkx as nx
# import numpy as np
# import matplotlib.pyplot as plt
# import random, time

# TODO: Load REAL network into G
# G = ...

# TODO: Print stats and components summary
# print(...)


In [1]:
import networkx as nx
import pandas as pd
import random

I have choosen the SNAP Social Circles Facebook dataset and unpacked the facebook_combined data into my github where I load below and create out graph that we will use throughout the project.

In [2]:



url = "https://raw.githubusercontent.com/nurfnick/NetworkScience/refs/heads/main/HomeworkAssignments/facebook_combined.txt"



G = nx.Graph()

df = pd.read_csv(url, header=None, sep=' ')
df.columns = ["node1", "node2"]


G.add_edges_from(df.values.tolist())

print(f"Number of nodes: {G.number_of_nodes()}")
print(f"Number of edges: {G.number_of_edges()}")
print(f"Density of Graph edges: { nx.density(G)}")
print(f"Number of components: {nx.number_connected_components(G)}")

Number of nodes: 4039
Number of edges: 88234
Density of Graph edges: 0.010819963503439287
Number of components: 1


I did no preprocessing to the data beyond unpacking the data.


## Part 1 — Graph Algorithms & Complexity (25%)
Implement:
1) Representations (edge list / adjacency list / adjacency matrix) with brief tradeoff discussion.
2) DFS/BFS: components; **triangle counting**; **diameter approximation** (explain method).
3) Shortest paths: **Dijkstra** from 5 sources; **Floyd–Warshall** APSP only if `|V| ≤ 1200`, else justify alternative.
4) **Betweenness centrality** (full or sampled, with sample size stated).
5) **Runtime study**: record times vs size (down‑sampled subgraphs) and relate to big‑O.


In [ ]:

# TODO: Implement components, triangle count, diameter approximation
# TODO: Implement Dijkstra (5 sources) and APSP strategy
# TODO: Compute betweenness (top‑5)
# TODO: Collect and present runtimes (table/figure) and commentary


I'll create the differing represnetations for edges.  
1. The edge list which was actually used to create the graph `G` is already here and loaded in a dataframe.  It makes it easy to find neighbors and also great for changing the structure of a network like adding or deleting edges.
2. Adjacency list is a linked list.  Again it is efficient for finding neighbors and graph traversal.
3. Adjacency matrix is the last structure and it is great for looking up neighbors but has large storage especially in networks with sparse edges.  Hard to delete edges for sure!

In [ ]:
count = 0
for i in G.adjacency():
  print(i)
  count += 1
  if count > 5:
    break


(0, {1: {}, 2: {}, 3: {}, 4: {}, 5: {}, 6: {}, 7: {}, 8: {}, 9: {}, 10: {}, 11: {}, 12: {}, 13: {}, 14: {}, 15: {}, 16: {}, 17: {}, 18: {}, 19: {}, 20: {}, 21: {}, 22: {}, 23: {}, 24: {}, 25: {}, 26: {}, 27: {}, 28: {}, 29: {}, 30: {}, 31: {}, 32: {}, 33: {}, 34: {}, 35: {}, 36: {}, 37: {}, 38: {}, 39: {}, 40: {}, 41: {}, 42: {}, 43: {}, 44: {}, 45: {}, 46: {}, 47: {}, 48: {}, 49: {}, 50: {}, 51: {}, 52: {}, 53: {}, 54: {}, 55: {}, 56: {}, 57: {}, 58: {}, 59: {}, 60: {}, 61: {}, 62: {}, 63: {}, 64: {}, 65: {}, 66: {}, 67: {}, 68: {}, 69: {}, 70: {}, 71: {}, 72: {}, 73: {}, 74: {}, 75: {}, 76: {}, 77: {}, 78: {}, 79: {}, 80: {}, 81: {}, 82: {}, 83: {}, 84: {}, 85: {}, 86: {}, 87: {}, 88: {}, 89: {}, 90: {}, 91: {}, 92: {}, 93: {}, 94: {}, 95: {}, 96: {}, 97: {}, 98: {}, 99: {}, 100: {}, 101: {}, 102: {}, 103: {}, 104: {}, 105: {}, 106: {}, 107: {}, 108: {}, 109: {}, 110: {}, 111: {}, 112: {}, 113: {}, 114: {}, 115: {}, 116: {}, 117: {}, 118: {}, 119: {}, 120: {}, 121: {}, 122: {}, 123: 

In [ ]:
nx.adjacency_matrix(G).todense()

array([[0, 1, 1, ..., 0, 0, 0],
       [1, 0, 0, ..., 0, 0, 0],
       [1, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

Now onto components.  We have only one component here.  We have already seen that only one component exists.  We utilize the built ins to compute the number of triangles.  Interestingly we see that 0 has lots of triangles but 11 has none!

In [ ]:
list(nx.triangles(G).items())[0:12]

[(0, 2519),
 (1, 57),
 (2, 40),
 (3, 86),
 (4, 39),
 (5, 26),
 (6, 14),
 (7, 82),
 (8, 19),
 (9, 634),
 (10, 37),
 (11, 0)]

For diameter, I do the following with a builtin.  This was LONG computation time!!!  I think I'll explore this and try to do a different algorithm.

In [ ]:
nx.diameter(G)

8

So I am going to randomly pick some nodes and find the node they are furthest from.  I'll update my diameter everytime I find one that is the furthest.

In [ ]:
dist = 0
numedges = 10
l = random.choices(list(G.nodes()), k=numedges)

for edge in l:
  for node in list(G.nodes()):
    p = nx.shortest_path_length(G, source=edge, target=node)
    if p > dist:
      dist = p

dist

8

Um, I found the longest in several iterations of this!  That's fairly random...  I find this algorithm clunky.  I could do much better if I built the tree for the 10 nodes I picked and then look at the length of it.  I think this algorithm is building that tree for each of the nodes.  This is a clunky way to do it but kinda works.  This algorithm is $O(n^2)*O(shortestpath)$.  I am going to improve on this by creating a breadth first algorithm.

In [30]:
def bfs(node):
  dist = [0]*(G.number_of_nodes())
  q = [node]
  visited = [False]*(G.number_of_nodes())
  visited[node] = True
  while q:
    u = q.pop(0)
    for v in G.neighbors(u):
      if not visited[v]:
        visited[v] = True
        dist[v] = dist[u] + 1
        q.append(v)
  return max(dist)

bfs(1010)

6

Now I use this to repeat teh experiment for the diameter.

In [35]:
dist = 0
numedges = 10
l = random.choices(list(G.nodes()), k=numedges)

for edge in l:
  p = bfs(edge)
  if p > dist:
    dist = p

dist

7

I do get a bit of variety when repeating this experiment but it is generally close to 8.  The big O for bredth fist is O(V + E).  For diameter then it will be O(V(V+E)).  We do that here below

In [36]:
for edge in G.nodes():
  p = bfs(edge)
  if p > dist:
    dist = p

dist

8


## Part 2 — Connectivity & Flow (10%)
- 3 source–sink pairs: **max‑flow** and **min‑cut**.
- 5 random pairs: **k** vertex‑ or edge‑disjoint paths for k∈{2,3}.

**Output:** table of results + 1–2 sentence robustness interpretation.


In [ ]:

# TODO: Max‑flow / min‑cut for 3 pairs
# TODO: k‑disjoint paths checks for 5 pairs (k=2,3)
# TODO: Summarize in a table



## Part 3 — Measurement Error & Link Prediction (20%)
- Remove **10%** of edges (seeded) to form `G_obs`; optionally add ≤2% spurious edges.
- Implement at least three methods: Jaccard, Adamic–Adar or Preferential Attachment, **Katz (truncated)** or **Resource Allocation**.
- Evaluate with a balanced test set (removed edges vs sampled non‑edges). Report **AUC** and **Precision@k** (`k∈{50,100}`) and include one evaluation plot.


In [ ]:

# TODO: Build G_obs with controlled edge removals/additions
# TODO: Implement similarity scores; assemble candidate pairs
# TODO: Create labeled test set; compute AUC and Precision@k
# TODO: Plot one ROC‑style or Precision@k figure



## Part 4 — Real‑World Structure Analysis (25%)
- **Degree distributions** (log–log); estimate power‑law exponent via MLE or compare to lognormal; state method.
- **Centrality distributions**: degree, betweenness, closeness, eigenvector (figures + discussion).
- **Clustering**: global and distribution of local clustering.
- **Assortativity**: degree assortativity coefficient and interpretation.


In [ ]:

# TODO: Degree distribution plots + parameter estimation or model comparison
# TODO: Centrality distribution plots + brief discussion
# TODO: Clustering metrics and local clustering distribution
# TODO: Degree assortativity value and interpretation



## Part 5 — Random Graph Models & Comparison (15%)
Generate matched models and compare to the real graph:
- **ER/gnp** with `p ≈ 2m / (n(n−1))`
- **Preferential Attachment (BA)** with parameters approximating |E|
- **Configuration Model** using the real graph’s degree sequence (project to simple graph; note any changes)

Report for each: `|V|`, `|E|`, avg degree, **avg path length** (LCC), **clustering**, degree distribution (overlay OK). Provide a comparison table and short commentary.


In [ ]:

# TODO: Generate ER, BA, and Configuration graphs
# TODO: Compute requested statistics and plots
# TODO: Comparison table + commentary



## Part 6 — Mini Write‑up (2–3 pages) (5%)
Synthesize results across parts; cite datasets and any extra packages; note limitations (scalability, sampling, model assumptions).



## Rubric (100 pts)
- Part 1 Algorithms & Complexity — **25**
- Part 2 Connectivity & Flow — **10**
- Part 3 Measurement Error & Link Prediction — **20**
- Part 4 Real‑World Structure — **25**
- Part 5 Random Models Comparison — **15**
- Part 6 Write‑up — **5**

**Integrity & Reproducibility**
- Student‑written code required; prohibited libraries → 0 for affected parts.
- Notebook must run end‑to‑end; missing citations → −2 to −3.
- Include **LLM Usage Log** below.


## LLM Usage Log

In [ ]:
# TODO: Paste 2–3 prompts and notes on what was accepted vs modified.
